<a href="https://colab.research.google.com/github/MikaTan2007/bc-wildfire-prediction/blob/main/bc_wildfire_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BC Wildfire Prediction Model

### Setup

In [1]:
!pip install geemap
!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.0 MB/s eta 0:00:00


In [2]:
import ee
import geemap
import geopandas as gpd
import pandas as pd

ee.Authenticate()

ee.Initialize(project='bc-wildfire-491217')

### Defining British Columbia

In [3]:
bc_bounds = ee.Geometry.BBox(-139.059399, 48.232274, -114.077738, 60.001826) #Bounds of BC
map = geemap.Map(center=[54.0, -125.0], zoom=5)

map.add_basemap('SATELLITE')
map.add_basemap('TERRAIN')

map.addLayer(bc_bounds, {'color': 'blue'}, 'BC Bounding Box') #Blue Filled Box
map.addLayer(ee.Image().paint(bc_bounds, 0, 0.5), {}, 'BC Outline Box') #Outline Box

map

Map(center=[54.0, -125.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

### ECMWF/ERA5/DAILY Dataset

In [ ]:
beginning_date = '2015-07-01'
end_date = '2015-07-31'

era5 = (
    ee.ImageCollection("ECMWF/ERA5/DAILY")
    .select(['mean_2m_air_temperature', 'total_precipitation'])
    .filterBounds(bc_bounds)
    .filterDate(beginning_date, end_date)
)

# Calculate the mean weather over BC for that entire month
mean_weather = era5.mean().clip(bc_bounds)

# Calculate the single average temperature for the WHOLE province
mean_scalar = mean_weather.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=bc_bounds,
    scale=27830 # ERA5 scale is roughly 27.8km
)

# .getInfo() forces Earth Engine to compute it and return a Python dictionary
result = mean_scalar.getInfo()

# Convert Kelvin to Celsius
temp_c = result['mean_2m_air_temperature'] - 273.15
print(f"Average BC Temperature for {beginning_date} to {end_date}: {temp_c:.2f}°C")

map = geemap.Map(center=[54.0, -125.0], zoom=5)
temp_viz = {
    'min': 273.15,
    'max': 303.15,
    'palette': [
        '#00008F', '#0000FF', '#0080FF', '#00FFFF', '#7FFF7F',
        '#FFFF00', '#FFBF00', '#FF8000', '#FF0000', '#800000'
    ]
}

map.add_basemap('SATELLITE')
map.add_basemap('TERRAIN')

map.addLayer(
    mean_weather.select('mean_2m_air_temperature'),
    temp_viz,
    f'Mean Temperature {beginning_date} to {end_date}'
)

map.addLayer(ee.Image().paint(bc_bounds, 0, 0.5), {}, 'BC Outline Box') #Outline Box

map

Average BC Temperature for 2015-07-01 to 2015-07-31: 14.45°C


Map(center=[54.0, -125.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

### ERA5-Land Daily Dataset

In [ ]:
beginning_date = '2015-07-01'
end_date = '2015-07-31'
# Load ERA5-Land Daily dataset
era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .select(['temperature_2m', 'total_precipitation_sum']) # Updated band names
    .filterBounds(bc_bounds)
    .filterDate(beginning_date, end_date)
)

# Calculate the mean weather over BC for July
mean_weather = era5.mean().clip(bc_bounds)

# Calculate the single average temperature for the WHOLE province
mean_scalar = mean_weather.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=bc_bounds,
    scale=11132 # ERA5 scale is roughly 11.1km
)

# .getInfo() forces Earth Engine to compute it and return a Python dictionary
result = mean_scalar.getInfo()

# Convert Kelvin to Celsius
temp_c = result['temperature_2m'] - 273.15
print(f"Average BC Temperature for {beginning_date} to {end_date}: {temp_c:.2f}°C")

map = geemap.Map(center=[54.0, -125.0], zoom=5)
temp_viz = {
    'min': 273.15,
    'max': 303.15,
    'palette': [
        '#00008F', '#0000FF', '#0080FF', '#00FFFF', '#7FFF7F',
        '#FFFF00', '#FFBF00', '#FF8000', '#FF0000', '#800000'
    ]
    #'palette': ['blue', 'white', 'red'] or ['blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}

map.add_basemap('SATELLITE')
map.add_basemap('TERRAIN')

map.addLayer(
    mean_weather.select('temperature_2m'),
    temp_viz,
    f'Mean Temperature {beginning_date} to {end_date}'
)

map.addLayer(ee.Image().paint(bc_bounds, 0, 0.5), {}, 'BC Outline Box') #Outline Box

map

Average BC Temperature for 2015-07-01 to 2015-07-31: 14.59°C


Map(center=[54.0, -125.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

### Historical Wildfire Dataset

In [ ]:
import os
import urllib.request
import zipfile

url = "https://cwfis.cfs.nrcan.gc.ca/downloads/nfdb/fire_pnt/current_version/NFDB_point.zip"
zip_path = "NFDB_point.zip"
extract_dir = "NFDB_point_data"

print("Downloading CWFIS National Fire Database...")
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Files extracted to: {extract_dir}/")
print(os.listdir(extract_dir))

Files extracted to: NFDB_point_data/
['NFDB_point_20250519.shx', 'NFDB_point_shapefile_metadata.pdf', 'NFDB_Point_metadata_NAP_ISO_19115_2003_FR.pdf', 'NFDB_point_20250519_stats.xlsx', 'NFDB_point_20250519.sbx', 'NFDB_point_20250519.shp.xml', 'NFDB_point_20250519.dbf', 'NFDB_Point_metadata_NAP_ISO_19115_2003_EN.pdf', 'NFDB_point_20250519.cpg', 'NFDB_Point_metadata_NAP_ISO_19115_2003.xml', 'NFDB_point_20250519.sbn', 'NFDB_point_20250519.shp', 'NFDB_point_20250519.prj']


In [ ]:
shp_file = os.path.join(extract_dir, "NFDB_point.shp")

print("Loading Shapefile into GeoPandas...")
gdf = gpd.read_file("NFDB_point_data/NFDB_point_20250519.shp") #NFDB_point_20250519.shp

Loading Shapefile into GeoPandas...


,NFDBFIREID,SRC_AGENCY,NAT_PARK,FIRE_ID,FIRENAME,LATITUDE,LONGITUDE,YEAR,MONTH,DAY,...,RESPONSE,PROTZONE,PRESCRIBED,MORE_INFO,CFS_NOTE1,CFS_NOTE2,ACQ_DATE,layer,omit,geometry
0,AB-2024-CWF-001-2024,AB,None,CWF-001-2024,None,50.066333,-114.154883,2024,1,2,...,None,None,None,Calgary Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-1345400.786 322485.621 0)
1,AB-2024-HWF-001-2024,AB,None,HWF-001-2024,None,57.912833,-116.334050,2024,1,5,...,None,None,None,High Level Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-1210041.546 1182823.657 0)
2,AB-2024-SWF-001-2024,AB,None,SWF-001-2024,None,56.575550,-115.216533,2024,1,17,...,None,None,None,Slave Lake Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-1194304.684 1023994.495 0)
3,AB-2024-LWF-001-2024,AB,None,LWF-001-2024,None,55.957361,-110.709667,2024,1,9,...,None,None,None,Lac La Biche Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-950764.968 883261.867 0)
4,AB-2024-LWF-002-2024,AB,None,LWF-002-2024,None,55.957361,-110.709667,2024,1,9,...,None,None,None,Lac La Biche Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-950764.968 883261.867 0)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
442398,BC-2024-2024-G90777,BC,None,2024-G90777,G90777,59.946100,-123.888600,2024,6,4,...,FUL,None,None,None,"NOTE overwinter fire, rep area burned in 2024",None,2025-04-08,BC_NFDB_2024,None,POINT Z (-1516114.145 1548789.258 0)
442399,BC-2024-2024-G90207,BC,None,2024-G90207,Patry Creek,59.310300,-123.282500,2024,7,16,...,FUL,None,None,None,"NOTE overwinter fire, rep area burned in 2024",None,2025-04-08,BC_NFDB_2024,None,POINT Z (-1516080.541 1472302.048 0)
442400,BC-2024-2024-G90324,BC,None,2024-G90324,G90324,59.313900,-121.988500,2024,7,10,...,MOD,None,None,None,"NOTE overwinter fire, rep area burned in 2024",None,2025-04-08,BC_NFDB_2024,None,POINT Z (-1450882.688 1442470.922 0)
442401,BC-2024-2024-G90285,BC,None,2024-G90285,G90285,59.789200,-121.606500,2024,7,10,...,MOD,None,None,None,"NOTE overwinter fire, rep area burned in 2024",None,2025-04-08,BC_NFDB_2024,None,POINT Z (-1410645.101 1480874.311 0)


In [ ]:
gdf

,NFDBFIREID,SRC_AGENCY,NAT_PARK,FIRE_ID,FIRENAME,LATITUDE,LONGITUDE,YEAR,MONTH,DAY,...,RESPONSE,PROTZONE,PRESCRIBED,MORE_INFO,CFS_NOTE1,CFS_NOTE2,ACQ_DATE,layer,omit,geometry
0,AB-2024-CWF-001-2024,AB,None,CWF-001-2024,None,50.066333,-114.154883,2024,1,2,...,None,None,None,Calgary Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-1345400.786 322485.621 0)
1,AB-2024-HWF-001-2024,AB,None,HWF-001-2024,None,57.912833,-116.334050,2024,1,5,...,None,None,None,High Level Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-1210041.546 1182823.657 0)
2,AB-2024-SWF-001-2024,AB,None,SWF-001-2024,None,56.575550,-115.216533,2024,1,17,...,None,None,None,Slave Lake Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-1194304.684 1023994.495 0)
3,AB-2024-LWF-001-2024,AB,None,LWF-001-2024,None,55.957361,-110.709667,2024,1,9,...,None,None,None,Lac La Biche Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-950764.968 883261.867 0)
4,AB-2024-LWF-002-2024,AB,None,LWF-002-2024,None,55.957361,-110.709667,2024,1,9,...,None,None,None,Lac La Biche Forest Area,None,None,2025-04-07,AB_NFDB_2024,None,POINT Z (-950764.968 883261.867 0)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
442398,BC-2024-2024-G90777,BC,None,2024-G90777,G90777,59.946100,-123.888600,2024,6,4,...,FUL,None,None,None,"NOTE overwinter fire, rep area burned in 2024",None,2025-04-08,BC_NFDB_2024,None,POINT Z (-1516114.145 1548789.258 0)
442399,BC-2024-2024-G90207,BC,None,2024-G90207,Patry Creek,59.310300,-123.282500,2024,7,16,...,FUL,None,None,None,"NOTE overwinter fire, rep area burned in 2024",None,2025-04-08,BC_NFDB_2024,None,POINT Z (-1516080.541 1472302.048 0)
442400,BC-2024-2024-G90324,BC,None,2024-G90324,G90324,59.313900,-121.988500,2024,7,10,...,MOD,None,None,None,"NOTE overwinter fire, rep area burned in 2024",None,2025-04-08,BC_NFDB_2024,None,POINT Z (-1450882.688 1442470.922 0)
442401,BC-2024-2024-G90285,BC,None,2024-G90285,G90285,59.789200,-121.606500,2024,7,10,...,MOD,None,None,None,"NOTE overwinter fire, rep area burned in 2024",None,2025-04-08,BC_NFDB_2024,None,POINT Z (-1410645.101 1480874.311 0)


### Filtering for fires in BC in or later than 2010

In [ ]:
# Filter for fires managed/reported by BC, in or later than 2010
bc_fires = gdf[(gdf['SRC_AGENCY'] == 'BC') & (gdf['YEAR'] >= 2010)].copy()

columns_to_keep = ['YEAR', 'MONTH', 'DAY', 'LATITUDE', 'LONGITUDE', 'SIZE_HA', 'CAUSE']
bc_fires_cleaned = bc_fires[columns_to_keep]

print(f"Successfully isolated {len(bc_fires_cleaned)} historical fires in BC!")
display(bc_fires_cleaned)

dataframe = pd.DataFrame(bc_fires_cleaned)
dataframe.to_csv('bc_fires.csv', index=False)

bc_fires_cleaned.to_csv("bc_historical_fires.csv", index=False)

Successfully isolated 22619 historical fires in BC!


,YEAR,MONTH,DAY,LATITUDE,LONGITUDE,SIZE_HA,CAUSE
13192,2023,9,27,51.8197,-122.0729,0.010,H
13193,2023,6,2,54.5692,-128.4611,1.400,H
13194,2023,6,27,50.1166,-119.2738,0.009,H
13195,2023,4,28,54.2270,-125.6543,1.100,H
13196,2023,4,17,49.5187,-119.5348,0.009,H
...,...,...,...,...,...,...,...
442398,2024,6,4,59.9461,-123.8886,306.730,N
442399,2024,7,16,59.3103,-123.2825,170806.900,N
442400,2024,7,10,59.3139,-121.9885,0.000,N
442401,2024,7,10,59.7892,-121.6065,0.000,N


### Extracting Geographical Features for those areas in GEE

In [4]:
historical_fires = pd.read_csv('bc_historical_fires.csv')
historical_fires

,YEAR,MONTH,DAY,LATITUDE,LONGITUDE,SIZE_HA,CAUSE
0,2023,9,27,51.8197,-122.0729,0.010,H
1,2023,6,2,54.5692,-128.4611,1.400,H
2,2023,6,27,50.1166,-119.2738,0.009,H
3,2023,4,28,54.2270,-125.6543,1.100,H
4,2023,4,17,49.5187,-119.5348,0.009,H
...,...,...,...,...,...,...,...
22614,2024,6,4,59.9461,-123.8886,306.730,N
22615,2024,7,16,59.3103,-123.2825,170806.900,N
22616,2024,7,10,59.3139,-121.9885,0.000,N
22617,2024,7,10,59.7892,-121.6065,0.000,N


In [5]:
ee_points = []

for index, row in historical_fires.iterrows():
  # Creating a GEE Feature
  point = ee.Feature(
      ee.Geometry.Point([
          row['LONGITUDE'],
          row['LATITUDE']
      ]),
      {
          'YEAR': int(row['YEAR']),
          'MONTH': int(row['MONTH']),
          'DAY': int(row['DAY']),
          'SIZE_HA': float(row['SIZE_HA']),
          'LATITUDE': row['LATITUDE'],
          'LONGITUDE': row['LONGITUDE'],
          'label': 1 # Positive wildfire sample
      }
  )
  ee_points.append(point)

In [6]:
fire_features = ee.FeatureCollection(ee_points)
print(f"Loaded {fire_features.size().getInfo()} fires into GEE")

Loaded 22619 fires into GEE


In [7]:
def extract_weather_for_fire(feature):
  # Get date of the specific fire
  year = feature.get('YEAR')
  month = feature.get('MONTH')
  day = feature.get('DAY')

  date_start = ee.Date.fromYMD(year, month, day)
  date_end = date_start.advance(1, 'day')

  #Obtain ERA5-Land Daily Dataset Info
  daily_weather = (
      ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
      .filterDate(date_start, date_end)
      .select(['temperature_2m', 'total_precipitation_sum', 'dewpoint_temperature_2m'])
      .first()
  )

  sampled = daily_weather.reduceRegion(
      reducer = ee.Reducer.first(),
      geometry = feature.geometry(),
      scale = 11132
  )

  return feature.set(sampled)

In [8]:
# test_subset = fire_features.limit(100)

processed_subset = fire_features.map(extract_weather_for_fire)

bc_weather_fires = geemap.ee_to_df(processed_subset)

In [9]:
# Post-processing: convert Kelvin to Celsius
bc_weather_fires['temperature_c'] = bc_weather_fires['temperature_2m'] - 273.15
bc_weather_fires['dewpoint_c'] = bc_weather_fires['dewpoint_temperature_2m'] - 273.15

bc_weather_fires = bc_weather_fires.drop(columns=['dewpoint_temperature_2m', 'temperature_2m'])

In [10]:
display(bc_weather_fires.head())

,DAY,LATITUDE,LONGITUDE,MONTH,SIZE_HA,YEAR,label,total_precipitation_sum,temperature_c,dewpoint_c
0,27,51.8197,-122.0729,9,0.010,2023,1,1.548231e-05,8.709819,1.660377
1,2,54.5692,-128.4611,6,1.400,2023,1,5.963278e-04,9.751162,3.830872
2,27,50.1166,-119.2738,6,0.009,2023,1,9.160727e-04,20.171532,8.554643
3,28,54.2270,-125.6543,4,1.100,2023,1,8.523463e-07,6.832194,-3.269081
4,17,49.5187,-119.5348,4,0.009,2023,1,5.518898e-03,-0.026009,-4.410338


In [11]:
bc_weather_fires.to_csv("bc_weather_fires.csv", index=False)

### Random Negative Points

In [28]:
import random

landcover = ee.ImageCollection('MODIS/061/MCD12Q1').select('LC_Type1').mode()
land_mask = landcover.neq(17) # 17 is water

land_only_image = ee.Image(1).updateMask(land_mask)

num_points = int(22619 * 1.75)

raw_random_points = ee.FeatureCollection.randomPoints(
    region = bc_bounds,
    points = num_points,
    seed = 42,
)

print(f"Generated {raw_random_points.size().getInfo()} random points")

Generated 39583 random points


In [29]:
land_only_points = land_mask.reduceRegions(
    collection = raw_random_points,
    reducer = ee.Reducer.first(),
    scale = 500
)

In [30]:
final_land_points = land_only_points.filter(ee.Filter.eq('first', 1))
negative_features = final_land_points.limit(22619)
print(f"Trimmed land points down to {negative_features.size().getInfo()}")

Trimmed land points down to 22619


In [35]:
def assign_random_dates(feature):
  year = random.randint(2010, 2024)
  month = random.randint(4,10)
  day = random.randint(1,28)

  geom = feature.geometry()

  longitude = geom.coordinates().get(0)
  latitude = geom.coordinates().get(1)

  return feature.set({
      'YEAR': year,
      'MONTH': month,
      'DAY': day,
      'LATITUDE': latitude,
      'LONGITUDE': longitude,
      'label': 0 # No fire
  })

negative_features = negative_features.map(assign_random_dates)
print(f"Generated {negative_features.size().getInfo()} random negative sample dates")

Generated 22619 random negative sample dates


#### Generating Weather Features for Negative Points

In [36]:
processed_negatives = negative_features.map(extract_weather_for_fire)

In [37]:
# Convert to Pandas
weather_negatives = geemap.ee_to_df(processed_negatives)

display(weather_negatives)

,DAY,LATITUDE,LONGITUDE,MONTH,YEAR,dewpoint_temperature_2m,first,label,temperature_2m,total_precipitation_sum
0,19,55.461379,-123.491434,10,2024,274.872904,1,0,275.334607,0.015847
1,19,56.280860,-122.315891,10,2024,272.377787,1,0,273.699272,0.004848
2,19,55.631288,-129.770640,10,2024,273.025085,1,0,273.283337,0.016898
3,19,55.583015,-123.960178,10,2024,273.399190,1,0,274.077608,0.012424
4,19,55.406582,-127.247938,10,2024,271.130391,1,0,272.136283,0.012208
...,...,...,...,...,...,...,...,...,...,...
22614,19,59.835931,-123.779892,10,2024,265.078552,1,0,269.298148,0.003726
22615,19,55.103353,-125.395793,10,2024,274.527445,1,0,274.949678,0.011275
22616,19,52.762550,-120.655334,10,2024,271.095723,1,0,271.969372,0.015322
22617,19,49.340171,-118.248329,10,2024,274.735046,1,0,275.262585,0.004880


In [41]:
# Post-processing: convert Kelvin to Celsius
weather_negatives['temperature_c'] = weather_negatives['temperature_2m'] - 273.15
weather_negatives['dewpoint_c'] = weather_negatives['dewpoint_temperature_2m'] - 273.15

weather_negatives = weather_negatives.drop(columns=['dewpoint_temperature_2m', 'temperature_2m'])

KeyError: 'temperature_2m'

In [42]:
weather_negatives = weather_negatives.drop(columns = ['first'])

In [43]:
weather_negatives

,DAY,LATITUDE,LONGITUDE,MONTH,YEAR,label,total_precipitation_sum,temperature_c,dewpoint_c
0,19,55.461379,-123.491434,10,2024,0,0.015847,2.184607,1.722904
1,19,56.280860,-122.315891,10,2024,0,0.004848,0.549272,-0.772213
2,19,55.631288,-129.770640,10,2024,0,0.016898,0.133337,-0.124915
3,19,55.583015,-123.960178,10,2024,0,0.012424,0.927608,0.249190
4,19,55.406582,-127.247938,10,2024,0,0.012208,-1.013717,-2.019609
...,...,...,...,...,...,...,...,...,...
22614,19,59.835931,-123.779892,10,2024,0,0.003726,-3.851852,-8.071448
22615,19,55.103353,-125.395793,10,2024,0,0.011275,1.799678,1.377445
22616,19,52.762550,-120.655334,10,2024,0,0.015322,-1.180628,-2.054277
22617,19,49.340171,-118.248329,10,2024,0,0.004880,2.112585,1.585046


In [ ]:
weather_negatives.to_csv("bc_weather_negatives.csv", index=False)